<div style="padding: 28px 32px;
            font-family: 'Georgia', serif;">

  <p style="margin:0; font-size:13px; letter-spacing:3px; text-transform:uppercase; color:#b0477d;">
     Statistics for Data Analytics · CA1 
  </p>

  <h1 style="margin:8px 0 4px 0; font-size:32px;">
    Payment Behaviour at Cosmovision
  </h1>
  <p style="margin:0 0 18px 0; font-size:16px; font-style:italic; color:#7a4a6a;">
    A statistical analysis of customer arrears and disconnection risk
  </p>

  <p style="margin:4px 0; font-size:15px;"><b>Student:</b> María Fernanda López Díaz</p>
  <p style="margin:4px 0; font-size:15px;"><b>Student ID:</b> 2026794</p>
  <p style="margin:4px 0; font-size:15px;"><b>Lecturer:</b> Taufique Ahmed</p>
  <p style="margin:4px 0; font-size:15px;"><b>GitHub:</b>
    <a href="https://github.com/mafs794/late-payment-prediction" style="color:#b0477d;">
      github.com/mafs794/late-payment-prediction</a></p>
</div>

In [12]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
EXPORT_DATE = pd.Timestamp("2026-02-10")

In [ ]:
# --- Rename columns from spanish to english ---
english_columns = ["id", "status", "area", "sub_area", "collector", "plan", "fee", "ad_tvs", "ad_fee", "balance", "+nov_balance", "dic_balance", 
                   "jan_balance", "feb_balance", "registration_date", "termination_date", "reconnection_date", "last_payment_date", "installation_day", 
                   "payday", "year_paid", "month_paid", "visit_date"]
raw_data = pd.read_excel("data/raw/data.xlsx", sheet_name="Sheet1", names=english_columns, header=1)

<div style="border-left:6px solid #f06fa8; border-radius:12px;
            padding:16px 22px; margin-top:24px; font-family:'Georgia', serif;">
  <h2 style="margin:0 0 6px 0; color:#7a4a6a;">1. Data Preparation</h2>
  <p style="margin:0; font-size:15px; line-height:1.6;">
    In this section I load the anonymised February 2026 service report, check data types 
    and missing values, and derive new variables such as tenure, days since last payment
    and months owed, all calculated relative to the export date.
  </p>
</div>

In [9]:
raw_data.head()

,id,status,area,sub_area,collector,plan,fee,ad_tvs,ad_fee,balance,...,feb_balance,registration_date,termination_date,reconnection_date,last_payment_date,installation_day,payday,year_paid,month_paid,visit_date
0,100,ALTA,4 - ALDEA BUENA VISTA,13 - CALLE PRINCIPAL,mario,429 - INTERNET 60 MEGAS FO CON CABLE TV,199,0,0,199,...,199,28/11/2023,NaN,NaN,04/02/2026,28,4,2026,1,NaN
1,101,ALTA,1 - MILPAS ALTAS,4 - MAGDALENA,mario,376 - INTERNET 25 MEGAS FO CON CABLE TV FO,199,0,0,376,...,199,02/02/2021,05/02/2021,09/02/2021,21/01/2026,9,21,2025,12,NaN
2,102,ALTA,1 - MILPAS ALTAS,4 - MAGDALENA,mario,6 - INTERNET 10 MEGAS FIBRA OPTICA CON CABLE T...,150,0,0,0,...,0,02/02/2021,NaN,NaN,05/02/2026,2,5,2026,2,NaN
3,103,ALTA,1 - MILPAS ALTAS,4 - MAGDALENA,mario,6 - INTERNET 10 MEGAS FIBRA OPTICA CON CABLE T...,150,0,0,150,...,150,01/03/2021,NaN,NaN,24/01/2026,1,24,2026,1,NaN
4,104,ALTA,1 - MILPAS ALTAS,4 - MAGDALENA,daniel,6 - INTERNET 10 MEGAS FIBRA OPTICA CON CABLE T...,150,0,0,0,...,0,02/03/2021,NaN,NaN,05/02/2026,2,5,2026,2,NaN


In [11]:
process_data = raw_data.copy()    

for col in ["registration_date", "reconnection_date"]:
    process_data[col] = pd.to_datetime(process_data[col], dayfirst=True, errors="coerce")

# Tenure counts from reinstatement if the customer came back, otherwise from installation
process_data["registration_date"] = process_data["reconnection_date"].fillna(process_data["registration_date"])
process_data["tenure_months"] = ((EXPORT_DATE - process_data["registration_date"]).dt.days / 30.44).round(1)

# Checks
print(process_data["tenure_months"].describe())
print("Missing:", process_data["tenure_months"].isna().sum())
print("Reinstated customers:", process_data["reconnection_date"].notna().sum())

count    1013.000000
mean       16.516979
std        12.706680
min         0.000000
25%         6.500000
50%        14.200000
75%        24.100000
max        60.200000
Name: tenure_months, dtype: float64
Missing: 0
Reinstated customers: 56


In [13]:
# --- Days since last payment ---
process_data["last_payment_date"] = pd.to_datetime(process_data["last_payment_date"], dayfirst=True, errors="coerce")
process_data["days_since_payment"] = (EXPORT_DATE - process_data["last_payment_date"]).dt.days

#Checks
print(process_data["days_since_payment"].describe())
print("Missing:", process_data["days_since_payment"].isna().sum())

count    1002.000000
mean       24.411178
std        41.068893
min         0.000000
25%         6.000000
50%        20.000000
75%        30.000000
max       860.000000
Name: days_since_payment, dtype: float64
Missing: 11


In [18]:
# usar el 100 por que es el minimo de mensualidad al mes, el precio del servicio de cable 
process_data[(process_data["balance"] > 0) & (process_data["balance"] < 100)].shape[0]

2

In [22]:
# --- Months Owed ---
process_data["monthly_fee"] = process_data["fee"] + process_data["ad_fee"].fillna(0)
process_data["months_owed"] = np.where(
    process_data["balance"] > 0,                              
    np.ceil(process_data["balance"] / process_data["monthly_fee"]),
    0
).astype(int)

print(process_data["months_owed"].value_counts().sort_index())
print("Missing:", process_data["months_owed"].isna().sum())
print("Fee = 0:", (process_data["monthly_fee"] == 0).sum())


months_owed
0    313
1    463
2    199
3     21
4     14
5      3
Name: count, dtype: int64
Missing: 0
Fee = 0: 14


In [24]:
process_data.shape

(1013, 27)